# ДЗ №1 — Классическое машинное обучение: классификация и регрессия

**Один датасет — две задачи.** Предскажем качество огранки (`high_cut`, классификация) и цену (`price_usd`, регрессия).

**Инструкция:** запустите ячейки сверху вниз; в местах **TODO** допишите код и удалите соответствующую строку `raise NotImplementedError(...)`. Помните: **обе целевые переменные запрещено использовать как входные признаки**. Разделение train/val/test уже подготовлено, не меняйте его.

Для работы потребуется `diamonds_hw01.csv` в папке ноутбука. Если вы используете Colab, загрузите CSV на вкладке **Files**. Подробности — `jupyter_environment_guide.md`.

## 0. Импортируем библиотеки

Импорты уже готовы. Для запуска в Colab, как правило, установка дополнительных библиотек не нужна.

In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_STATE = 42

## 1. Знакомство с данными — K1

Загрузите файл `diamonds_hw01.csv`, посмотрите первые строки и распределение целей. Обратите внимание: в `carat`, `depth_pct` и `clarity` есть пропуски.

In [ ]:
file_path = Path("diamonds_hw01.csv")
if not file_path.is_file():
    raise FileNotFoundError(
        "Не найден diamonds_hw01.csv. Положите CSV рядом с ноутбуком; "
        "в Google Colab загрузите его через панель Files."
    )

df = pd.read_csv(file_path)
print("Размер:", df.shape)
display(df.head())

In [ ]:
print("Типы и пропуски:")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
print("Классы high_cut:")
display(df["high_cut"].value_counts())
print("Распределение цены:")
display(df["price_usd"].describe())

**TODO — ваш комментарий (2–3 предложения).** Какие признаки числовые, какие категориальные? Какие пропуски нужно обработать? Как распределена целевая переменная классификации?

*Напишите свой ответ здесь.*

## 2. Формируем признаки и разделяем данные

Код готов. Мы удаляем **обе цели** из `X`, чтобы не допустить утечку. Дальше делим строки на train (60%), validation (20%) и test (20%) — **одинаково для обеих задач**.

In [ ]:
FEATURES_NUM = ["carat", "depth_pct", "table_pct", "length_mm", "width_mm", "height_mm"]
FEATURES_CAT = ["color", "clarity"]
FEATURES = FEATURES_NUM + FEATURES_CAT

X = df[FEATURES]
y_cls = df["high_cut"]
y_reg = df["price_usd"]

X_trainval, X_test, yc_trainval, yc_test, yr_trainval, yr_test = train_test_split(
    X, y_cls, y_reg, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cls
)
X_train, X_val, yc_train, yc_val, yr_train, yr_val = train_test_split(
    X_trainval, yc_trainval, yr_trainval,
    test_size=0.25, random_state=RANDOM_STATE, stratify=yc_trainval
)

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

## 3. Предобработка — K2

Соберите два пайплайна: для числовых признаков — **медиана + стандартизация**, для категориальных — **мода + One-Hot Encoding**. Затем объедините их в `ColumnTransformer`. Здесь нужно дописать **6 коротких строк**.

In [ ]:
numeric_pipeline = Pipeline([
    # TODO: заполнить пропуски медианой через SimpleImputer
    # TODO: стандартизировать через StandardScaler
])

categorical_pipeline = Pipeline([
    # TODO: заполнить пропуски самым частым значением через SimpleImputer
    # TODO: применить OneHotEncoder(handle_unknown="ignore")
])

preprocessor = ColumnTransformer([
    # TODO: добавить числовой пайплайн и список FEATURES_NUM
    # TODO: добавить категориальный пайплайн и список FEATURES_CAT
])

assert len(numeric_pipeline.steps) == 2, "TODO: добавьте два шага обработки числовых признаков"
assert len(categorical_pipeline.steps) == 2, "TODO: добавьте два шага обработки категориальных признаков"
assert len(preprocessor.transformers) == 2, "TODO: объедините числовую и категориальную обработку"

## 4. Классификация — K3

Включите четыре модели. Для каждой соберите **свой** `Pipeline` из предобработки и классификатора. Сравните `Accuracy_val`, `F1_train`, `F1_val`. Лучший результат по `F1_val` — наибольший.

In [ ]:
cls_models = {
    "DummyClassifier": DummyClassifier(strategy="most_frequent"),
    # TODO: LogisticRegression(max_iter=1000)
    # TODO: DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
    # TODO: RandomForestClassifier(n_estimators=80, max_depth=12,
    #                              random_state=RANDOM_STATE, n_jobs=-1)
}
assert len(cls_models) == 4, "TODO: добавьте ещё три модели классификации"

In [ ]:
cls_rows = []

for name, classifier in cls_models.items():
    pipe = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(classifier)),
    ])

    # TODO: обучите pipe на X_train и yc_train
    # TODO: предскажите метки для X_train и X_val
    # TODO: заполните словарь с ключами Model, Accuracy_val, F1_train, F1_val
    # Подсказка: f1_score(..., average="macro")
    raise NotImplementedError("TODO: напишите обучение и расчёт метрик классификации")

    # cls_rows.append({
    #     "Model": name,
    #     "Accuracy_val": ...,
    #     "F1_train": ...,
    #     "F1_val": ...,
    # })

## 5. Сравнительная таблица классификации — K5

Эта ячейка готова: после выполнения предыдущего TODO она сформирует и отсортирует таблицу.

In [ ]:
cls_comparison = (
    pd.DataFrame(cls_rows)
    .sort_values("F1_val", ascending=False)
    .reset_index(drop=True)
)
assert len(cls_comparison) == 4, "В таблице должны быть все четыре классификатора"
display(cls_comparison.round(3))

## 6. Регрессия — K4

Построим **отдельную** таблицу с регрессорами. Для MAE меньше — лучше; для R² больше — лучше. Предобработка и разбиение данных остаются теми же.

In [ ]:
reg_models = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    # TODO: LinearRegression()
    # TODO: DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)
    # TODO: RandomForestRegressor(n_estimators=80, max_depth=12,
    #                             random_state=RANDOM_STATE, n_jobs=-1)
}
assert len(reg_models) == 4, "TODO: добавьте ещё три модели регрессии"

In [ ]:
reg_rows = []

for name, regressor in reg_models.items():
    pipe = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(regressor)),
    ])

    # TODO: обучите pipe на X_train и yr_train
    # TODO: получите прогнозы цены для X_train и X_val
    # TODO: добавьте словарь с Model, MAE_train, MAE_val, R2_val
    # Подсказка: mean_absolute_error(y_true, y_pred); r2_score(y_true, y_pred)
    raise NotImplementedError("TODO: напишите обучение и расчёт метрик регрессии")

    # reg_rows.append({
    #     "Model": name,
    #     "MAE_train": ...,
    #     "MAE_val": ...,
    #     "R2_val": ...,
    # })

## 7. Сравнительная таблица регрессии — K5

Отсортируем модели по **возрастанию** `MAE_val`. Помните, что MAE измеряется в долларах США.

In [ ]:
reg_comparison = (
    pd.DataFrame(reg_rows)
    .sort_values("MAE_val", ascending=True)
    .reset_index(drop=True)
)
assert len(reg_comparison) == 4, "В таблице должны быть все четыре регрессора"
display(reg_comparison.round(2))

## 8. Финальная проверка — K5

Выберите победителя по validation, обучите заново на `train + validation` и измерьте качество на `test` **один раз**. Для классификации — F1-macro, для регрессии — MAE. Не выбирайте победителей по тесту.

In [ ]:
best_cls_name = cls_comparison.iloc[0]["Model"]
final_cls = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", clone(cls_models[best_cls_name])),
])

# TODO: final_cls.fit(X_trainval, yc_trainval)
# TODO: cls_test_pred = final_cls.predict(X_test)
# TODO: cls_test_f1 = f1_score(yc_test, cls_test_pred, average="macro")
raise NotImplementedError("TODO: финальная проверка классификации")

print(f"Лучшая классификация: {best_cls_name}; F1_test={cls_test_f1:.3f}")

In [ ]:
best_reg_name = reg_comparison.iloc[0]["Model"]
final_reg = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", clone(reg_models[best_reg_name])),
])

# TODO: final_reg.fit(X_trainval, yr_trainval)
# TODO: reg_test_pred = final_reg.predict(X_test)
# TODO: reg_test_mae = mean_absolute_error(yr_test, reg_test_pred)
raise NotImplementedError("TODO: финальная проверка регрессии")

print(f"Лучшая регрессия: {best_reg_name}; MAE_test=${reg_test_mae:.2f}")

## 9. Выводы — K6

**TODO — напишите 4–6 предложений:**

1. Какая модель лучше классифицирует высокое качество огранки? Насколько она лучше `DummyClassifier`?
2. Какая модель точнее предсказывает цену (в USD)? Насколько её MAE меньше, чем у `DummyRegressor`?
3. Есть ли признаки переобучения по разнице train/validation?
4. Каковы итоговые значения F1-macro и MAE на test?

*Ваши выводы здесь.*